# Executive Summary — DND Spring 2026 Datathon

**Team importNumpy** | Global Health Statistics Analysis

This notebook synthesizes findings from all 6 competition questions into a cohesive narrative, generates presentation-quality figures, and provides the formatted assumption log.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats as sp_stats

from notebooks._helpers import (
    get_connection, set_plot_style, save_fig,
    TIER_COLORS, COLOR_RURAL, COLOR_PERIURBAN, COLOR_URBAN,
    effect_size_label, format_ci, FIGURES_DIR,
)

set_plot_style()
con = get_connection()
print('Connected to DuckDB')

## 1. Summary Table: All Questions

In [ ]:
summary = pd.DataFrame([
    {
        'Question': 'Q1: Geographic Disparities',
        'Key Finding': 'No meaningful disparities (d < 0.01)',
        'Confidence': 'High',
        'Primary Metric': "Cohen's d, eta-squared",
        'Viz Reference': 'q1_bar_mortality_by_tier',
    },
    {
        'Question': 'Q2: Access vs Outcomes',
        'Key Finding': 'No relationship (R-sq < 0.001)',
        'Confidence': 'High',
        'Primary Metric': 'Pearson r, R-squared',
        'Viz Reference': 'q2_scatter_access_mortality',
    },
    {
        'Question': 'Q3: Outlier Communities',
        'Key Finding': 'No true outliers (< 0.5pp absolute diff)',
        'Confidence': 'Medium',
        'Primary Metric': 'Z-scores, defiance index',
        'Viz Reference': 'q3_quadrant_scatter',
    },
    {
        'Question': 'Q4: Sensitivity Analysis',
        'Key Finding': 'Insensitive to tier definitions (< 0.04pp)',
        'Confidence': 'High',
        'Primary Metric': 'Tier spread, eta-squared',
        'Viz Reference': 'q4_3panel_tier_schemes',
    },
    {
        'Question': 'Q5: Sparse Reporting',
        'Key Finding': '640/640 groups adequate; massively overpowered',
        'Confidence': 'High',
        'Primary Metric': 'Power analysis, CI width',
        'Viz Reference': 'q5_forest_full_vs_sparse',
    },
    {
        'Question': 'Q6: Premature Conclusions',
        'Key Finding': 'Data is synthetic; 5 premature conclusions identified',
        'Confidence': 'High',
        'Primary Metric': 'Entropy, expected vs observed r',
        'Viz Reference': 'q6_expected_vs_observed',
    },
])

print(summary.to_string(index=False))

## 2. Dashboard Grid — Best Figure from Each Question

In [ ]:
# Generate compact versions of key figures for the dashboard

fig = plt.figure(figsize=(20, 13))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# ── Q1: Bar chart ──
ax1 = fig.add_subplot(gs[0, 0])
tier_data = con.execute("""
    SELECT urbanization_tier,
           AVG(avg_mortality) AS mort, STDDEV_SAMP(avg_mortality) AS sd,
           SUM(n) AS total_n
    FROM v_outcome_by_geography
    GROUP BY urbanization_tier
""").fetchdf()
tier_order = ['Rural', 'Peri-urban', 'Urban']
td = tier_data.set_index('urbanization_tier').reindex(tier_order)
ax1.bar(tier_order, td['mort'],
        color=[TIER_COLORS[t] for t in tier_order], edgecolor='white')
ax1.set_title('Q1: Mortality by Tier', fontweight='bold')
ax1.set_ylabel('Avg Mortality (%)')

# ── Q2: Scatter ──
ax2 = fig.add_subplot(gs[0, 1])
scatter_data = con.execute("""
    SELECT healthcare_access, mortality_rate
    FROM clean_health WHERE data_quality_flag='ok'
    USING SAMPLE 5000
""").fetchdf()
ax2.scatter(scatter_data['healthcare_access'], scatter_data['mortality_rate'],
            alpha=0.1, s=5, color='#3498db')
ax2.set_title('Q2: Access vs Mortality', fontweight='bold')
ax2.set_xlabel('Healthcare Access (%)')
ax2.set_ylabel('Mortality (%)')

# ── Q3: Quadrant scatter ──
ax3 = fig.add_subplot(gs[0, 2])
outlier_data = con.execute('SELECT country, z_access, z_mortality FROM v_outlier_communities').fetchdf()
ax3.scatter(outlier_data['z_access'], outlier_data['z_mortality'],
            s=60, color='#2c3e50', edgecolors='white')
for _, r in outlier_data.iterrows():
    ax3.annotate(r['country'], (r['z_access'], r['z_mortality']),
                textcoords='offset points', xytext=(5, 3), fontsize=6)
ax3.axhline(0, color='gray', linewidth=0.5)
ax3.axvline(0, color='gray', linewidth=0.5)
ax3.set_title('Q3: Country Outlier Map', fontweight='bold')
ax3.set_xlabel('z(Access)')
ax3.set_ylabel('z(Mortality)')

# ── Q4: Sensitivity bars ──
ax4 = fig.add_subplot(gs[1, 0])
sens_data = con.execute('SELECT * FROM v_sensitivity_tiers ORDER BY tier_scheme, tier_label').fetchdf()
for scheme, color in [('default_3tier', '#3498db'), ('alt1_binary', '#e74c3c'), ('alt2_4tier', '#2ecc71')]:
    subset = sens_data[sens_data['tier_scheme'] == scheme]
    ax4.plot(range(len(subset)), subset['avg_mortality'].values, 'o-',
             label=scheme, color=color, markersize=6)
ax4.set_title('Q4: Sensitivity Across Schemes', fontweight='bold')
ax4.set_ylabel('Avg Mortality (%)')
ax4.legend(fontsize=7)

# ── Q5: CI forest ──
ax5 = fig.add_subplot(gs[1, 1])
country_ci = con.execute("""
    SELECT country,
           AVG(mortality_rate) AS m,
           1.96*STDDEV_SAMP(mortality_rate)/SQRT(COUNT(*)) AS ci
    FROM clean_health WHERE data_quality_flag='ok'
    GROUP BY country ORDER BY AVG(mortality_rate)
""").fetchdf()
ax5.errorbar(country_ci['m'], range(len(country_ci)), xerr=country_ci['ci'],
             fmt='o', color='#2c3e50', markersize=4, capsize=2)
ax5.set_yticks(range(len(country_ci)))
ax5.set_yticklabels(country_ci['country'], fontsize=7)
ax5.set_title('Q5: Mortality CIs by Country', fontweight='bold')
ax5.set_xlabel('Mortality (%)')

# ── Q6: Expected vs Observed ──
ax6 = fig.add_subplot(gs[1, 2])
sample_corr = con.execute("""
    SELECT mortality_rate, healthcare_access, per_capita_income,
           education_index, urbanization_rate, doctors_per_1000, recovery_rate
    FROM clean_health WHERE data_quality_flag='ok'
    USING SAMPLE 10000
""").fetchdf()
cm = sample_corr.corr()
pairs_exp = [
    ('Access→Mort', -0.45, float(cm.loc['mortality_rate', 'healthcare_access'])),
    ('Income→Access', 0.60, float(cm.loc['per_capita_income', 'healthcare_access'])),
    ('Edu→Mort', -0.40, float(cm.loc['education_index', 'mortality_rate'])),
]
x = np.arange(len(pairs_exp))
ax6.bar(x - 0.15, [p[1] for p in pairs_exp], 0.3, label='Expected', color='#3498db')
ax6.bar(x + 0.15, [p[2] for p in pairs_exp], 0.3, label='Observed', color='#e74c3c')
ax6.set_xticks(x)
ax6.set_xticklabels([p[0] for p in pairs_exp], fontsize=8)
ax6.set_title('Q6: Expected vs Observed r', fontweight='bold')
ax6.legend(fontsize=7)
ax6.axhline(0, color='black', linewidth=0.5)

save_fig(fig, 'summary_dashboard_grid')
plt.show()

## 3. Impact-Confidence Scatter

In [ ]:
impact_data = pd.DataFrame([
    {'question': 'Q1', 'label': 'Geographic\nDisparities', 'impact': 8, 'confidence': 9},
    {'question': 'Q2', 'label': 'Access vs\nOutcomes', 'impact': 9, 'confidence': 9},
    {'question': 'Q3', 'label': 'Outlier\nCommunities', 'impact': 5, 'confidence': 7},
    {'question': 'Q4', 'label': 'Sensitivity\nAnalysis', 'impact': 6, 'confidence': 9},
    {'question': 'Q5', 'label': 'Sparse\nReporting', 'impact': 7, 'confidence': 9},
    {'question': 'Q6', 'label': 'Premature\nConclusions', 'impact': 10, 'confidence': 9},
])

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for i, row in impact_data.iterrows():
    ax.scatter(row['confidence'], row['impact'], s=300, c=colors[i],
               edgecolors='white', linewidth=2, zorder=5)
    ax.annotate(f'{row["question"]}\n{row["label"]}',
                (row['confidence'], row['impact']),
                textcoords='offset points', xytext=(15, 0),
                fontsize=9, va='center')

ax.set_xlabel('Confidence in Finding (1-10)', fontsize=12)
ax.set_ylabel('Impact / Importance (1-10)', fontsize=12)
ax.set_title('Executive Summary: Impact vs Confidence for All 6 Questions')
ax.set_xlim(4, 11)
ax.set_ylim(3, 11)
ax.axhline(7, color='gray', linestyle='--', alpha=0.3)
ax.axvline(7, color='gray', linestyle='--', alpha=0.3)

save_fig(fig, 'summary_impact_confidence')
plt.show()

## 4. Assumption Log

In [ ]:
# Load and display the assumption log
import csv
from pathlib import Path

assumption_path = Path('..') / 'reports' / 'assumption_log.csv'
if assumption_path.exists():
    assumptions = pd.read_csv(assumption_path)
    print(f'Assumption log: {len(assumptions)} entries')
    print(assumptions.to_string(index=False))
else:
    print('Assumption log not found — run pipeline first.')

## 5. Overall Narrative

### The Meta-Finding

The Global Health Statistics dataset is **synthetically generated** using uniform random distributions with no embedded relationships between variables. This means:

1. **No health disparities exist** between geographic tiers — not because equity has been achieved, but because the data generator used the same distribution for all groups
2. **No access-outcome relationship** — healthcare access was generated independently of mortality/recovery
3. **No country-level variation** — all 20 countries are statistically identical
4. **No confounders** — because no real causal structure exists

### Why This Matters for the Competition

Teams that force insights from this data will produce misleading conclusions. Our approach:
- **Honest analysis** — we report what the data actually shows
- **Effect sizes over p-values** — with N~1M, everything is "significant" but nothing is meaningful
- **Structured framework** — our premature-conclusions framework demonstrates analytical maturity
- **Simulated sparsity** — we show competence with uncertainty concepts even when the data doesn't require it

### Recommendations (if this were real data)

If a similar dataset with REAL data showed these patterns, we would recommend:
1. Investigate data collection methodology for systematic biases
2. Check for data aggregation artifacts that mask true disparities
3. Supplement with qualitative data from health workers on the ground
4. Conduct targeted studies in specific regions before making policy decisions

---

## 6. Dual-Track Synthesis

The **Benchmark Comparison notebook** (`benchmark_comparison.ipynb`) extends this analysis with a second track showing what real-world health data would reveal based on WHO/GBD literature ranges.

In [ ]:
# Dual-track summary table with benchmark comparison column
import matplotlib.patches as mpatches

dual_summary = pd.DataFrame([
    {
        'Question': 'Q1: Geographic Disparities',
        'Observed Finding': 'No disparities (d < 0.01)',
        'Benchmark Expectation': '5-15pp rural-urban gap',
        'Gap Factor': '~500x',
    },
    {
        'Question': 'Q2: Access vs Outcomes',
        'Observed Finding': 'No relationship (r ~ 0)',
        'Benchmark Expectation': 'r ~ -0.45',
        'Gap Factor': '~900x',
    },
    {
        'Question': 'Q3: Outlier Communities',
        'Observed Finding': 'No outliers (< 0.5pp)',
        'Benchmark Expectation': '13pp country spread',
        'Gap Factor': '~26x',
    },
    {
        'Question': 'Q4: Sensitivity Analysis',
        'Observed Finding': 'Insensitive (< 0.04pp)',
        'Benchmark Expectation': '5-9pp scheme spread',
        'Gap Factor': '~150x',
    },
    {
        'Question': 'Q5: Sparse Reporting',
        'Observed Finding': '0% groups insufficient',
        'Benchmark Expectation': '30-60% groups insufficient',
        'Gap Factor': 'N/A',
    },
    {
        'Question': 'Q6: Premature Conclusions',
        'Observed Finding': '0/8 significant correlations',
        'Benchmark Expectation': '6-7/8 significant',
        'Gap Factor': 'N/A',
    },
])

print('DUAL-TRACK SUMMARY — Observed vs Benchmark')
print('=' * 80)
print(dual_summary.to_string(index=False))
print('=' * 80)
print()
print('See benchmark_comparison.html for 8 detailed comparison figures')
print('and the 30/60/90-day action plan.')

In [ ]:
# ── Evidence Ladder (claim discipline) ────────────────────────────────────
from IPython.display import display
from notebooks._helpers import load_synthetic_signatures

evidence_ladder = pd.DataFrame([
    {'Level': 'Descriptive', 'Meaning': 'What the provided dataset contains', 'Not allowed': 'Causal claims / policy prescriptions'},
    {'Level': 'Robustness', 'Meaning': 'Stability under sensitivity checks', 'Not allowed': 'Cherry-picked definitions / hidden transforms'},
    {'Level': 'Illustrative', 'Meaning': 'Benchmarks/simulations to show expected signal', 'Not allowed': 'Mixing benchmark into observed results'},
])
print('\nEVIDENCE LADDER — how to read our claims')
display(evidence_ladder)

# ── Synthetic signature headline metrics (machine-readable) ────────────────
sig = load_synthetic_signatures()
if sig:
    coverage = sig.get('combination_coverage', {})
    corr = sig.get('correlation_scan', {})
    label = sig.get('label_reliability', {})

    headline = pd.DataFrame([
        {'metric': 'Rows', 'value': sig.get('n_rows')},
        {'metric': 'Combo coverage rate', 'value': coverage.get('coverage_rate')},
        {'metric': 'Duplicate groups (n>1)', 'value': sig.get('duplicate_groups', {}).get('count_groups_with_n_gt_1')},
        {'metric': 'Max |correlation| (scan)', 'value': corr.get('max_abs_corr')},
        {'metric': 'Mean entropy (disease_category_original within disease)', 'value': label.get('entropy_by_disease', {}).get('mean')},
        {'metric': "Cramer's V (disease_name vs original category)", 'value': label.get('independence', {}).get('cramers_v')},
    ])
    print('\nSYNTHETIC SIGNATURES — headline metrics (from reports/synthetic_signatures.json)')
    display(headline)
else:
    print('\nSynthetic signatures JSON not found. Run the pipeline to generate reports/synthetic_signatures.json.')

# ── Robustness: base vs dedup-at-cell-grain ────────────────────────────────
try:
    robust = con.execute(
        'SELECT * FROM robustness_delta_summary ORDER BY view_name'
    ).fetchdf()
    print('\nROBUSTNESS SUMMARY — base vs dedup-at-cell-grain')
    display(robust)
except Exception as e:
    print(f'\nRobustness summary not available: {e}')

con.close()
print('Executive summary complete.')
print(f'All figures saved to: {FIGURES_DIR}')